## Structured Output

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

from langchain.chat_models import init_chat_model
model=init_chat_model("groq:llama-3.3-70b-versatile")      ## you can use chatgroq() also


### 1)pydantic

In [11]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="the year of the movie")
    director:str=Field(description="the director of the movie")

model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000254BFC2A510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000254BFC2B230>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {

In [7]:
model.invoke("provide details about the bolloywood movie Animal")

AIMessage(content='"Animal" is an upcoming Indian Hindi-language action thriller film directed by Sandeep Reddy Vanga, who is known for his work on the Telugu film "Arjun Reddy" and its Hindi remake "Kabir Singh." The movie is set to release on August 11, 2023.\n\nHere are some key details about the film:\n\n**Cast:**\nThe film features an ensemble cast, including:\n- Ranbir Kapoor as the protagonist\n- Rashmika Mandanna\n- Anil Kapoor\n- Bobby Deol\n- Triptii Dimri\n\n**Plot:**\nWhile the exact plot of the film is not yet fully disclosed, it is reported to be an action-packed thriller that explores themes of family, power, and loyalty. Ranbir Kapoor is expected to play the lead role of a complex character with a dark past.\n\n**Production:**\nThe movie is produced by Bhushan Kumar\'s T-Series, Krishan Kumar, Murad Khetani, and Ashwin Varde\'s Cine1 Studios. The film\'s music is composed by Harshavardhan Rameshwar and Pritam.\n\n**Filming:**\nPrincipal photography for the film began in

In [12]:
model_with_structure.invoke("provide details about the bolloywood movie Animal")

Movie(title='Animal', year=2023, director='Sandeep Reddy Vanga')

In [15]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str
    year:int
    director:str
    cast:list[str]

model_with_structure=model.with_structured_output(Movie)
model_with_structure.invoke("provide details about the south movie DJ")

Movie(title='Duvvada Jagannadham', year=2017, director='Harish Shankar', cast=['Allu Arjun', 'Pooja Hegde'])

In [22]:

from langchain.agents import create_agent

agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Movie
)

res = model.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "extract info from Movie ADI, year=4000, directed by john and cast is leo"
            }
        ]
    }
)

res

{'messages': [HumanMessage(content='extract info from Movie ADI, year=4000, directed by john and cast is leo', additional_kwargs={}, response_metadata={}, id='1c80d035-bba6-496f-99a2-070355a2aa93'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '90vjbfc1h', 'function': {'arguments': '{"cast":["leo"],"director":"john","title":"ADI","year":4000}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 261, 'total_tokens': 291, 'completion_time': 0.077805051, 'completion_tokens_details': None, 'prompt_time': 0.015183049, 'prompt_tokens_details': None, 'queue_time': 0.16114791, 'total_time': 0.0929881}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ecfa0-e94a-74c0-b1a6-3541caf4dfe0-0', tool_calls=[{'name': 'Movie', 'args': {'cast': ['leo'], 'director': 'john

In [23]:
res["structured_response"]

Movie(title='ADI', year=4000, director='john', cast=['leo'])

### 2) Dataclass

In [26]:
from dataclasses import dataclass

@dataclass
class Movie:
    title:str
    year:int
    director:str

agent=create_agent(
    model="groq:llama-3.3-70b-versatile",
    response_format=Movie
)

res = model.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "extract info from Movie ADI, year=4000, directed by john and cast is leo"
            }
        ]
    }
)

res["structured_response"]

Movie(title='ADI', year=4000, director='john', cast=['leo'])

InvalidUpdateError: Expected dict, got provide details about the bolloywood movie Animal
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE